# *Análisis de métricas de negocio con SQL: entendiendo el desempeño comercial del videoclub Sakila*


## 🧩 1. Contexto del negocio
El negocio **Sakila Rental** es una cadena de renta de DVDs que ha observado fluctuaciones en sus ingresos y quiere **entender el comportamiento de sus clientes y sucursales**.  
Tiene información de clientes, películas, pagos, inventario y empleados distribuidos en varias tiendas.  
Busca tomar decisiones informadas sobre **qué géneros, clientes o sucursales generan mayor valor** y cómo optimizar su rentabilidad.

---


## 💡 2. Problemática general  
> Las ventas han disminuido en los últimos meses, y la dirección quiere entender **qué factores influyen más en los ingresos**: el tipo de película, la frecuencia de renta, la ubicación de la tienda o el desempeño del staff.  

---

## 🗣️ 3. Preguntas del stakeholder  
1. ¿Cuáles son las películas o géneros más rentables y con mayor rotación?  
2. ¿Qué tiendas y empleados tienen mejor desempeño en ventas?  
3. ¿Qué tipo de clientes generan más ingresos (por país, antigüedad, frecuencia)?  
4. ¿Existen patrones temporales en los ingresos (por mes, día, fin de semana)?  
5. ¿Qué estrategias podrían aumentar el valor del cliente o mejorar la eficiencia operativa?

---


## 📏 4. KPIs de interés  

| KPI | Descripción | Métrica SQL |
|------|--------------|-------------|
| **Revenue total** | Suma total de pagos realizados | `SUM(payment.amount)` |
| **Número de rentas** | Volumen total de transacciones | `COUNT(rental_id)` |
| **Ticket promedio (ARPU)** | Promedio de gasto por cliente | `SUM(amount)/COUNT(DISTINCT customer_id)` |
| **Top categorías por ingreso** | Géneros más rentables | `GROUP BY category.name` |
| **Ingresos por empleado** | Ventas promedio por staff | `SUM(amount) GROUP BY staff_id` |
| **Ingresos por tienda** | Ranking de tiendas por facturación | `SUM(amount) GROUP BY store_id` |
| **Retención de clientes** | Clientes activos por mes | comparación mensual en `payment_date` |

---


## 🔍 5. Análisis propuesto


### 1. **Análisis general de ingresos y rentas**  
   - Query de ingresos totales, rentas y ticket promedio.  

#### 🧠 Queries propuestos


**1. Ingresos totales, número de rentas y ticket promedio**

```sql
SELECT 
    ROUND(SUM(p.amount), 2) AS total_revenue,
    COUNT(r.rental_id) AS total_rentals,
    ROUND(SUM(p.amount) / COUNT(DISTINCT r.customer_id), 2) AS avg_revenue_per_customer
FROM payment p
JOIN rental r ON p.rental_id = r.rental_id;
```
---


### 2. **Segmentación por géneros**  
   - Comparar categorías por ingresos y frecuencia de renta.  

#### 🧠 Queries propuestos


**2. Ingresos y frecuencia por categoría de película**

```sql
SELECT 
    c.name AS category,
    COUNT(r.rental_id) AS total_rentals,
    ROUND(SUM(p.amount), 2) AS total_revenue,
    ROUND(SUM(p.amount) / COUNT(r.rental_id), 2) AS avg_ticket
FROM payment p
JOIN rental r ON p.rental_id = r.rental_id
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film f ON i.film_id = f.film_id
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
GROUP BY c.name
ORDER BY total_revenue DESC;
```
---


### 3. **Desempeño por tienda y staff**  
   - Ranking de tiendas y empleados más productivos.  

#### 🧠 Queries propuestos


**3. Desempeño por tienda y empleado**

```sql
SELECT 
    s.store_id,
    CONCAT(st.first_name, ' ', st.last_name) AS staff_name,
    ROUND(SUM(p.amount), 2) AS total_revenue,
    COUNT(p.payment_id) AS transactions
FROM payment p
JOIN staff st ON p.staff_id = st.staff_id
JOIN store s ON st.store_id = s.store_id
GROUP BY s.store_id, staff_name
ORDER BY total_revenue DESC;
```
---


### 4. **Análisis temporal de ingresos**  
   - Tendencia mensual y estacionalidad.  

#### 🧠 Queries propuestos



**4. Tendencia mensual de ingresos**

```sql
SELECT 
    DATE_TRUNC('month', p.payment_date) AS month,
    ROUND(SUM(p.amount), 2) AS monthly_revenue,
    COUNT(p.payment_id) AS total_payments
FROM payment p
GROUP BY month
ORDER BY month;
```
---


### 5. **Segmentación de clientes**  
   - Identificación de clientes premium (por monto total o frecuencia).  

#### 🧠 Queries propuestos


**5. Clientes más valiosos (segmentación por ingresos)**

```sql
SELECT 
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    COUNT(r.rental_id) AS total_rentals,
    ROUND(SUM(p.amount), 2) AS total_spent
FROM payment p
JOIN rental r ON p.rental_id = r.rental_id
JOIN customer c ON r.customer_id = c.customer_id
GROUP BY c.customer_id, customer_name
ORDER BY total_spent DESC
LIMIT 10;
```
---

**6. Retención de clientes activos por mes**

```sql
SELECT 
    DATE_TRUNC('month', p.payment_date) AS month,
    COUNT(DISTINCT p.customer_id) AS active_customers
FROM payment p
GROUP BY month
ORDER BY month;
```
---



---

## 💬 6. Insights esperados  
- Las películas de **acción y comedia** concentran el mayor ingreso, pero también presentan alta rotación (inventario crítico).  
- La **tienda 1 supera a la tienda 2** en ingresos, pero no en número de clientes → oportunidad de expansión.  
- Los **clientes frecuentes generan el 60% de los ingresos** → oportunidad de fidelización.  
- Los **fines de semana concentran 40% de las rentas** → ajustar personal y campañas.  


## 🚀 7. Recomendaciones accionables  
1. Implementar un **programa de lealtad** para clientes frecuentes.  
2. **Optimizar inventario** priorizando géneros con mayor margen.  
3. Ajustar **horarios y staffing** en función del patrón temporal de rentas.  
4. **Incentivar empleados top** o replicar sus prácticas en otras tiendas.  

## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta  nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [`Discord`](https://discord.com/channels/1081207584104656986/1420849538196836472).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal [`#project`](https://discord.com/channels/1081207584104656986/1420848813186351134) para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
    - En tus preguntas recuerda etiquetar a `@Dataconsulta` y ubica tu pregunta de acuerdo a `Sprint/Capitulo/Seccion`
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨